## Funzione di ottimizzazione per l'evitamento dei limiti articolari

Nella cinematica inversa ridondante (robot con più gradi di libertà del
necessario), il sistema è **sotto-vincolato**: esistono infinite configurazioni
articolari che portano l'end-effector nella stessa posa. Questa libertà può
essere sfruttata per ottimizzare un criterio secondario.

In questa cella viene definita la funzione di ottimizzazione **w(q)**,
che penalizza le configurazioni in cui i giunti si avvicinano ai propri
limiti fisici. La funzione è costruita in modo che il suo gradiente ∂w/∂q
possa essere usato come termine di correzione nella cinematica inversa
(metodo dello **Jacobiano esteso** o **null-space projection**).

La formula utilizzata è:

$$w(\mathbf{q}) = -\frac{1}{2(n)} \sum_{i=1}^{n} \left(\frac{q_i - \bar{q}_i}{q_{i,max} - q_{i,min}}\right)^2$$

dove $\bar{q}_i$ è il centro del range articolare del giunto $i$.
Il segno negativo fa sì che **massimizzare w** equivalga a **stare lontano
dai limiti**. Il calcolo è fatto in modo **simbolico** con SymPy e poi
convertito in funzione NumPy tramite `lambdify` per un uso efficiente
a runtime.


In [2]:
import sympy as sym
from sympy import symbols, cos, sin, Matrix, simplify, sqrt, det, lambdify, rad
import dill
#Definizione delle variabili simboliche
pi = sym.pi
Qin = sym.Matrix([sym.rad(-129), sym.rad(-10), sym.rad(-86), sym.rad(0)]) #limiti inferiori dei giunti
Qf = sym.Matrix([sym.rad(25), sym.rad(165), sym.rad(11), sym.rad(360)]) #limiti superiori dei giunti
k = 4
C = []
for i in range(k): #calcola il valore medio tra i limiti inferiori e superiori dei giunti, che sarà il centro del range di movimento di ogni giunto
    C.append(((Qf[i]+Qin[i])/2))

q1, q2, q3, q4 = symbols('q1 q2 q3 q4', real=True)
Q = Matrix([q1, q2, q3, q4])

#4 gradi di libertà
n = 4

w = -1/(2*(n)) * sum(((Q[i] - C[i]) / (Qf[i] - Qin[i]))**2 for i in range(n)) #se considero il quarto giunto devo usare range(n)

W_q = []
for q in Q:
    W_q.append((w.diff(q)))  

#Tramite lambdify l'espressione simbolica appena definita è convertita in una funzione numpy 
W_q_numpy = sym.lambdify((Q), W_q, 'numpy')

print(W_q_numpy)
print(W_q)

#Salvataggio della funzione in un file esterno
with open('funzione_ottimizzazione_4giunti.pkl', 'wb') as file:
    dill.dump(W_q_numpy, file)

<function _lambdifygenerated at 0x11d138860>
[-0.170770787653905*(2*q1 + 26*pi/45)/pi**2, -0.132244897959184*(2*q2 - 31*pi/36)/pi**2, -0.430438941439048*(2*q3 + 5*pi/12)/pi**2, -0.03125*(2*q4 - 2*pi)/pi**2]


Come alternativa al funzionale definito sull’intero spazio dei giunti, è stato considerato un secondo caso in cui il criterio di ottimizzazione coinvolge esclusivamente i primi tre giunti. L’obiettivo è valutare se l’esclusione del quarto giunto, caratterizzato da una libertà di movimento illimitata, possa influenzare il comportamento del manipolatore durante l’esecuzione della traiettoria.

In [4]:
q1, q2, q3, q4 = symbols('q1 q2 q3 q4', real=True)
Q = Matrix([q1, q2, q3, q4])

n = 3

w = -1/(2*(n)) * sum(((Q[i] - C[i]) / (Qf[i] - Qin[i]))**2 for i in range(n)) 

W_q = []
for q in Q:
    W_q.append((w.diff(q)))  

#Tramite lambdify l'espressione simbolica appena definita è convertita in una funzione numpy 
W_q_numpy = sym.lambdify((Q), W_q, 'numpy')

print(W_q_numpy)
print(W_q)

#Salvataggio della funzione in un file esterno
with open('funzione_ottimizzazione_3giunti.pkl', 'wb') as file:
    dill.dump(W_q_numpy, file)

<function _lambdifygenerated at 0x11d13a340>
[-0.227694383538539*(2*q1 + 26*pi/45)/pi**2, -0.176326530612245*(2*q2 - 31*pi/36)/pi**2, -0.573918588585397*(2*q3 + 5*pi/12)/pi**2, 0]
